In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, classification_report
)

# Models
from sklearn.linear_model    import LogisticRegression
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.svm             import SVC
from sklearn.naive_bayes     import GaussianNB
from sklearn.ensemble        import RandomForestClassifier
from sklearn.tree            import DecisionTreeClassifier

# ─────────────────────────────────────────────
# 1. ENCODING
# ─────────────────────────────────────────────

df_model = pd.read_csv("../data/credit_risk_dataset_cleaned.csv")

# Ordinal encoding — loan_grade has a natural order (A=best → G=worst)
grade_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}
df_model["loan_grade"] = df_model["loan_grade"].map(grade_map)

# Binary encoding — Yes/No column
df_model["cb_person_default_on_file"] = (
    df_model["cb_person_default_on_file"]
    .map({"Y": 1, "N": 0})
)

# One-hot encoding — nominal categoricals with no order
df_model = pd.get_dummies(
    df_model,
    columns=["person_home_ownership", "loan_intent"],
    drop_first=True   # avoid dummy variable trap
)

print("Shape after encoding:", df_model.shape)
print("Columns:", df_model.columns.tolist())


# ─────────────────────────────────────────────
# 2. SPLIT
# ─────────────────────────────────────────────

X = df_model.drop("loan_status", axis=1)
y = df_model["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,        # preserve class ratio in both sets
    random_state=42
)

print(f"\nTrain size : {X_train.shape[0]}")
print(f"Test size  : {X_test.shape[0]}")
print(f"Default rate (train): {y_train.mean():.2%}")
print(f"Default rate (test) : {y_test.mean():.2%}")


# ─────────────────────────────────────────────
# 3. SCALING
# ─────────────────────────────────────────────
# Required for KNN, SVM, Logistic Regression, Naive Bayes
# Fit ONLY on train — transform both

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


# ─────────────────────────────────────────────
# 4. DEFINE MODELS
# ─────────────────────────────────────────────

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    "K-Nearest Neighbors": KNeighborsClassifier(
        n_neighbors=5,
        metric="euclidean"
    ),
    "SVM": SVC(
        kernel="rbf",
        probability=True,      # needed for roc_auc
        class_weight="balanced",
        random_state=42
    ),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
}

# Models that need scaled input
needs_scaling = {
    "Logistic Regression",
    "K-Nearest Neighbors",
    "SVM",
    "Naive Bayes",
}


# ─────────────────────────────────────────────
# 5. TRAIN + EVALUATE ALL MODELS
# ─────────────────────────────────────────────

results = []

for name, model in models.items():

    # Use scaled data for distance/linear models, raw for tree models
    if name in needs_scaling:
        X_tr, X_te = X_train_scaled, X_test_scaled
    else:
        X_tr, X_te = X_train.values, X_test.values

    # Train
    model.fit(X_tr, y_train)

    # Predict
    y_pred      = model.predict(X_te)
    y_pred_prob = model.predict_proba(X_te)[:, 1]

    # Cross-validation AUC (5-fold on training set)
    cv_auc = cross_val_score(
        model, X_tr, y_train,
        scoring="roc_auc", cv=5, n_jobs=-1
    ).mean()

    # Collect metrics
    results.append({
        "Model":       name,
        "Accuracy":    round(accuracy_score(y_test, y_pred),           4),
        "ROC-AUC":     round(roc_auc_score(y_test, y_pred_prob),       4),
        "Precision":   round(precision_score(y_test, y_pred),          4),
        "Recall":      round(recall_score(y_test, y_pred),             4),
        "F1-Score":    round(f1_score(y_test, y_pred),                 4),
        "CV AUC (5k)": round(cv_auc,                                   4),
    })

    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(classification_report(y_test, y_pred,
                                 target_names=["No Default", "Default"]))


# ─────────────────────────────────────────────
# 6. RESULTS TABLE
# ─────────────────────────────────────────────

results_df = pd.DataFrame(results).set_index("Model")

# Sort by ROC-AUC descending — the most important metric for credit risk
results_df = results_df.sort_values("ROC-AUC", ascending=False)

print("\n")
print("=" * 70)
print("  MODEL COMPARISON")
print("=" * 70)
print(results_df.to_string())
print("=" * 70)
print(f"\n  Best model : {results_df.index[0]}")
print(f"  Best AUC   : {results_df['ROC-AUC'].iloc[0]}")

Shape after encoding: (28800, 18)
Columns: ['person_age', 'person_income', 'person_emp_length', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']

Train size : 23040
Test size  : 5760
Default rate (train): 21.97%
Default rate (test) : 21.96%

  Logistic Regression
              precision    recall  f1-score   support

  No Default       0.92      0.79      0.85      4495
     Default       0.50      0.75      0.60      1265

    accuracy                           0.78      5760
   macro avg       0.71      0.77      0.73      5760
weighted avg       0.83      0.78      0.79      5760


  K-Nearest Neighbors
              precision    recall  f1-score   support

  No Default       0.89